# Teaching an AI to Land a Rocket in Gusty Wind

This notebook builds a 2-D rigid-body rocket simulator from scratch, integrated with RK4, and uses it to test whether a Deep Q-Network (DQN) agent trained with **domain randomisation** lands more reliably under unseen wind than a DQN trained in calm conditions or a classical PID controller. Success rates are estimated from 1,000 Monte Carlo test landings per controller, with 95% confidence intervals. The simulator is verified against free fall, the Tsiolkovsky rocket equation, conservation checks and the RK4 convergence order. [TODO: one-sentence key result.]

> **Work in progress.** The example code below (a Monte Carlo estimate of $\pi$) only shows the expected structure and will be replaced by the rocket model. The notebook must run with *Runtime → Run all* on a fresh Colab runtime. By default it loads pre-trained weights; set `RETRAIN = True` to train from scratch.


In [1]:
# Setup: install anything not already on Colab, then import
# !pip install simpy   # example
import numpy as np
import matplotlib.pyplot as plt

SEED = 42                      # state your seed(s) and keep results reproducible
rng = np.random.default_rng(SEED)
print("numpy", np.__version__)

numpy 2.5.5


## 1. Problem and question

Reusable rockets such as SpaceX's Falcon 9 land vertically under automated guidance, and reinforcement learning is being explored as an alternative to hand-tuned controllers. A learned controller trained in one simulated world may fail when conditions change, and this lack of robustness is a major barrier to trusting AI controllers in safety-critical systems.

**Question:** does training with domain randomisation (randomised mass, gravity and wind in every episode) make a DQN agent land more reliably under *unseen* wind conditions than (a) an agent trained in calm conditions and (b) a classical PID controller?


## 2. Model, assumptions and equations

**Assumptions** (full list and justification in `THEORY.md`)
1. Planar rigid body in the vertical $x$–$y$ plane; uniform gravity.
2. Main engine along the body axis; side thrusters give pure torque.
3. Quadratic drag on the velocity relative to a stochastic horizontal wind.
4. Synthetic, physically plausible parameters for a generic small lander.

**Equations** (state $(x, y, v_x, v_y, \theta, \omega, m)$, $\theta$ measured from vertical)

$$m\dot v_x = -T_m\sin\theta + F_{D,x}, \qquad m\dot v_y = T_m\cos\theta - mg + F_{D,y}, \qquad I(m)\,\dot\omega = \tau_s, \qquad \dot m = -\frac{T}{I_{sp}\,g_0}$$

$$\mathbf F_D = -\tfrac12\rho C_D A\,|\mathbf v_{rel}|\,\mathbf v_{rel}, \qquad \mathbf v_{rel} = (v_x - w,\; v_y), \qquad dw = -\frac{w-\bar w}{\tau_w}dt + \sigma_w\sqrt{2/\tau_w}\,dW_t$$

The equations are integrated with RK4.

**Concepts used:**
- reinforcement learning (DQN) and domain randomisation;
- numerical ODE solvers (RK4 and its error order);
- input modelling and random variate generation (wind gusts, randomised parameters);
- Monte Carlo estimation with confidence intervals (law of large numbers).


## 3. Implementation

### 3.1 Parameters

All values are synthetic and describe a generic small lander (units and reasons are in `THEORY.md`). "Unseen wind" means a mean wind of 10–15 m/s with gusts of 3–5 m/s; training never exceeds 8 m/s and 2 m/s. Mass and gravity stay nominal in the main test, so a drop in success rate comes from wind alone.

In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class RocketParams:
    """Nominal vehicle, environment and numerical parameters (SI units, angles in rad)."""
    m0: float = 1000.0        # kg    initial (wet) mass
    m_fuel: float = 100.0     # kg    usable fuel
    T_max: float = 18_000.0   # N     main-engine thrust (on/off)
    T_s: float = 400.0        # N     side-thruster thrust (on/off)
    l_s: float = 2.5          # m     side-thruster lever arm
    L: float = 6.0            # m     rocket length
    Isp: float = 220.0        # s     specific impulse
    g0: float = 9.80665       # m/s^2 standard gravity (defines Isp; never randomised)
    g: float = 9.81           # m/s^2 local gravity
    rho: float = 1.225        # kg/m^3 air density
    CdA: float = 5.0          # m^2   drag coefficient x reference area
    tau_w: float = 2.0        # s     gust correlation time
    dt: float = 0.1           # s     control and RK4 step
    t_max: float = 40.0       # s     episode time limit
    x_lim: float = 150.0      # m     flight area: |x| <= x_lim
    y_lim: float = 200.0      # m     flight area: y <= y_lim

    @property
    def m_dry(self):          # kg
        return self.m0 - self.m_fuel
    @property
    def twr(self):            # initial thrust-to-weight ratio
        return self.T_max / (self.m0 * self.g)
    @property
    def inertia0(self):       # kg m^2, uniform slender rod
        return self.m0 * self.L**2 / 12
    @property
    def max_steps(self):
        return round(self.t_max / self.dt)

@dataclass(frozen=True)
class LandingLimits:
    """Success: every limit holds at first ground contact (y <= 0)."""
    vy_max: float = 2.0               # m/s
    vx_max: float = 1.0               # m/s
    theta_max: float = np.deg2rad(10) # rad
    d_max: float = 10.0               # m, distance from pad centre
    x_pad: float = 0.0                # m

@dataclass(frozen=True)
class InitialConditions:
    """Uniform (low, high) ranges, used in every scenario."""
    x: tuple = (-30.0, 30.0)                              # m
    y: tuple = (80.0, 120.0)                              # m
    vx: tuple = (-3.0, 3.0)                               # m/s
    vy: tuple = (-12.0, -6.0)                             # m/s
    theta: tuple = (-np.deg2rad(5), np.deg2rad(5))        # rad
    omega: tuple = (-0.05, 0.05)                          # rad/s

@dataclass(frozen=True)
class Scenario:
    """Uniform ranges re-sampled at each reset; the sign of the mean wind is +/-1, p = 0.5."""
    name: str
    wind_abs: tuple = (0.0, 0.0)      # m/s   |mean wind|
    gust_sd: tuple = (0.0, 0.0)       # m/s   gust standard deviation
    m0: tuple = (1000.0, 1000.0)      # kg    wet mass (fuel load fixed)
    g: tuple = (9.81, 9.81)           # m/s^2

P, LIMITS, INIT = RocketParams(), LandingLimits(), InitialConditions()

CALM = Scenario("calm")                                                  # trains the calm DQN
DR_TRAIN = Scenario("domain randomisation", (0.0, 8.0), (0.0, 2.0),      # trains the robust DQN
                    (900.0, 1100.0), (9.3, 10.3))
UNSEEN = Scenario("unseen wind", (10.0, 15.0), (3.0, 5.0))               # main test: nominal m0 and g

N_TEST = 1000           # test episodes per controller
TEST_SEED = 2026        # same episodes for all three controllers
TRAIN_SEEDS = (0, 1, 2) # independent DQN training runs

In [3]:
# Plausibility checks on the parameters
twr_worst = P.T_max / (DR_TRAIN.m0[1] * DR_TRAIN.g[1])
alpha0 = P.T_s * P.l_s / P.inertia0
mdot = P.T_max / (P.Isp * P.g0)
dv = P.Isp * P.g0 * np.log(P.m0 / P.m_dry)
wind_acc = 0.5 * P.rho * P.CdA * UNSEEN.wind_abs[1]**2 / P.m0   # wind push on a rocket at rest

print(f"thrust-to-weight: {P.twr:.2f} nominal, {twr_worst:.2f} worst case")
print(f"inertia {P.inertia0:.0f} kg m^2, side-thruster angular acceleration {alpha0:.3f} rad/s^2")
print(f"main-engine burn {mdot:.2f} kg/s -> {P.m_fuel/mdot:.1f} s of thrust, delta-v {dv:.0f} m/s")
print(f"acceleration from a {UNSEEN.wind_abs[1]:.0f} m/s wind: {wind_acc:.2f} m/s^2 "
      f"(lean {np.rad2deg(np.arctan(wind_acc / P.g)):.1f} deg)")

assert twr_worst > 1.2, "cannot brake in the worst case"
assert dv > 2 * (abs(INIT.vy[0]) + P.g * 10), "too little fuel for a 10 s descent with a factor of 2"
assert np.arctan(wind_acc / P.g) < LIMITS.theta_max, "steady wind needs more lean than landing allows"
assert UNSEEN.wind_abs[0] > DR_TRAIN.wind_abs[1], "test mean wind overlaps training"
assert UNSEEN.gust_sd[0] > DR_TRAIN.gust_sd[1], "test gusts overlap training"
print("PASS: parameters are plausible and unseen wind lies outside the training range")

thrust-to-weight: 1.83 nominal, 1.59 worst case
inertia 3000 kg m^2, side-thruster angular acceleration 0.333 rad/s^2
main-engine burn 8.34 kg/s -> 12.0 s of thrust, delta-v 227 m/s
acceleration from a 15 m/s wind: 0.69 m/s^2 (lean 4.0 deg)
PASS: parameters are plausible and unseen wind lies outside the training range


### 3.2 Rigid-body dynamics and RK4

States are stored as a batch of shape $(n, 7)$, so many episodes advance in one NumPy call. Gravity and dry mass are passed per episode because domain randomisation varies them.

In [4]:
# Discrete actions: (main engine, side command); side: 0 none, +1 anticlockwise, -1 clockwise
ACTIONS = np.array([(u_m, u_s) for u_m in (0, 1) for u_s in (0, 1, -1)], dtype=float)
N_ACTIONS = len(ACTIONS)

def deriv(s, main, side, wind, g, m_dry, p=P):
    """ds/dt for a batch of states, shape (n, 7): x, y, vx, vy, theta, omega, m.
    main, side, wind, g and m_dry are scalars or arrays of shape (n,)."""
    x, y, vx, vy, th, om, m = s.T
    fuel_left = m > m_dry + 1e-9
    T = p.T_max * main * fuel_left                       # main thrust [N]
    S = p.T_s * side * fuel_left                         # signed side thrust [N]
    v_rel_x = vx - wind
    drag = 0.5 * p.rho * p.CdA * np.hypot(v_rel_x, vy)   # drag force / relative speed [kg/s]
    ax = (-T * np.sin(th) - drag * v_rel_x) / m
    ay = (T * np.cos(th) - drag * vy) / m - g
    alpha = S * p.l_s / (m * p.L**2 / 12)                # torque / I(m)
    mdot = -(T + np.abs(S)) / (p.Isp * p.g0)
    return np.stack([vx, vy, ax, ay, om, alpha, mdot], axis=1)

def rk4_step(s, main, side, wind, g, m_dry, dt=P.dt, p=P):
    """Classical fourth-order Runge-Kutta; controls and wind are held over the step."""
    f = lambda z: deriv(z, main, side, wind, g, m_dry, p)
    k1 = f(s)
    k2 = f(s + 0.5 * dt * k1)
    k3 = f(s + 0.5 * dt * k2)
    k4 = f(s + dt * k3)
    out = s + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    out[:, 6] = np.maximum(out[:, 6], m_dry)             # a step cannot burn fuel it does not have
    return out

def euler_step(s, main, side, wind, g, m_dry, dt=P.dt, p=P):
    """Forward Euler (first order), used only for comparison in the verification."""
    return s + dt * deriv(s, main, side, wind, g, m_dry, p)

### 3.3 Wind and episodes

The wind follows the exact Ornstein–Uhlenbeck update. An episode ends in success or crash at ground contact, or when the rocket leaves the flight area, runs out of fuel or exceeds the time limit.

In [5]:
FLYING, SUCCESS, CRASH, OUT_OF_AREA, OUT_OF_FUEL, TIMEOUT = range(6)
OUTCOMES = ["flying", "success", "crash", "out of area", "out of fuel", "timeout"]

def ou_step(w, w_mean, gust_sd, xi, dt=P.dt, tau=P.tau_w):
    """Exact one-step update of the Ornstein-Uhlenbeck wind; xi ~ N(0, 1)."""
    a = np.exp(-dt / tau)
    return w_mean + (w - w_mean) * a + gust_sd * np.sqrt(1 - a * a) * xi

def is_success(s, lim=LIMITS):
    """The landing definition, applied to states at ground contact."""
    return ((np.abs(s[:, 3]) <= lim.vy_max) & (np.abs(s[:, 2]) <= lim.vx_max)
            & (np.abs(s[:, 4]) <= lim.theta_max) & (np.abs(s[:, 0] - lim.x_pad) <= lim.d_max))

class RocketEnv:
    """n independent episodes advanced together. It knows nothing about controllers or rewards.

    Every step draws one gust per episode, including finished ones, so the same seed gives
    the same start states and winds whichever controller is flying."""

    def __init__(self, n, scenario, seed, p=P, init=INIT, lim=LIMITS):
        self.n, self.sc, self.p, self.init, self.lim = n, scenario, p, init, lim
        self.rng = np.random.default_rng(seed)
        self.s = np.zeros((n, 7))
        self.g, self.m_dry, self.w_mean, self.gust_sd, self.w = (np.zeros(n) for _ in range(5))
        self.k = np.zeros(n, dtype=int)
        self.outcome = np.zeros(n, dtype=int)
        self.reset(np.ones(n, dtype=bool))

    def reset(self, mask):
        """Start new episodes where mask is True; returns the states."""
        c, U, sc, i = int(mask.sum()), self.rng.uniform, self.sc, self.init
        m0 = U(*sc.m0, c)
        self.s[mask] = np.column_stack([U(*r, c) for r in (i.x, i.y, i.vx, i.vy, i.theta, i.omega)] + [m0])
        self.m_dry[mask] = m0 - self.p.m_fuel
        self.g[mask] = U(*sc.g, c)
        self.w_mean[mask] = self.rng.choice([-1.0, 1.0], c) * U(*sc.wind_abs, c)
        self.gust_sd[mask] = U(*sc.gust_sd, c)
        self.w[mask] = self.w_mean[mask] + self.gust_sd[mask] * self.rng.standard_normal(c)
        self.k[mask] = 0
        self.outcome[mask] = FLYING
        return self.s.copy()

    def step(self, action):
        """Advance unfinished episodes by one control step; returns (states, outcomes)."""
        live = self.outcome == FLYING
        main, side = ACTIONS[action].T
        old = self.s[live]
        new = rk4_step(old, main[live], side[live], self.w[live], self.g[live], self.m_dry[live], self.p.dt, self.p)

        # state at ground contact: linear interpolation to y = 0 within the last step
        hit = new[:, 1] <= 0
        f = old[hit, 1] / (old[hit, 1] - new[hit, 1])
        new[hit] = old[hit] + f[:, None] * (new[hit] - old[hit])

        self.k[live] += 1
        out = np.zeros(len(new), dtype=int)
        out[self.k[live] >= self.p.max_steps] = TIMEOUT
        out[new[:, 6] <= self.m_dry[live] + 1e-9] = OUT_OF_FUEL
        out[(np.abs(new[:, 0]) > self.p.x_lim) | (new[:, 1] > self.p.y_lim)] = OUT_OF_AREA
        out[hit] = np.where(is_success(new[hit], self.lim), SUCCESS, CRASH)

        self.s[live], self.outcome[live] = new, out
        self.w = ou_step(self.w, self.w_mean, self.gust_sd, self.rng.standard_normal(self.n), self.p.dt, self.p.tau_w)
        return self.s.copy(), self.outcome.copy()

## 5. Experiments and results

[Parameter study or scenario comparison that answers your question. Report replications/seeds and confidence intervals, not single runs.]

In [8]:
# REPLACE: your parameter sweep or scenario comparison

## 6. Discussion and limitations

[What the results mean for the real-world question; the most important assumptions and their effect; what you would do next.]

**Web demo reference value:** with default settings the web demo should reproduce [value ± tolerance] from section [N].

## 7. References

1. V. Mnih *et al.*, "Human-level control through deep reinforcement learning," *Nature*, vol. 518, no. 7540, pp. 529–533, 2015, doi: 10.1038/nature14236.
2. J. Tobin *et al.*, "Domain randomization for transferring deep neural networks from simulation to the real world," in *Proc. IEEE/RSJ IROS*, 2017, pp. 23–30, doi: 10.1109/IROS.2017.8202133.
3. R. S. Sutton and A. G. Barto, *Reinforcement Learning: An Introduction*, 2nd ed. MIT Press, 2018.

Full list: `REFERENCES.md`.
